In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("⏳ Khởi động Siêu máy tính Spark Single Node (16GB RAM)...")
start_time = time.time()

# Cấu hình Spark kết nối thẳng vào Iceberg Data Lakehouse
from pyspark.sql import SparkSession

# TỐI ƯU HÓA CHO MÁY ẢO 16GB RAM
spark = SparkSession.builder \
    .appName("Generate_Training_Data_Gold") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .config("spark.sql.shuffle.partitions", "32") \
    .config("spark.sql.catalog.gcp_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.gcp_catalog.type", "hadoop") \
    .config("spark.sql.catalog.gcp_catalog.warehouse", "gs://amazon-reviews-lakehouse-warehouse/warehouse/") \
    .getOrCreate()

print("✅ Đã kết nối Lakehouse. Bắt đầu trộn dữ liệu...")

# 1. Đọc Bảng Silver (Chứa lịch sử User đánh giá Item)
df_silver = spark.read.table("gcp_catalog.silver.amazon_reviews_cleaned")

# 2. Tạo Label (Nhãn): Nếu Rating >= 4 sao thì coi như Thích (1), ngược lại là Không thích (0)
df_interactions = df_silver.select(
    F.col("user_id_hashed").alias("user_id"),
    F.col("asin").alias("item_id"),
    F.when(F.col("rating") >= 4.0, 1.0).otherwise(0.0).alias("label")
)

# 3. Đọc Bảng Đặc trưng Người dùng (Gold Layer) đã tính trước đó
print("⏳ Đang nạp User Features...")
df_user_features = spark.read.parquet("gs://amazon-reviews-lakehouse-warehouse/warehouse/gold/user_features")

# 4. JOIN (Ghép) dữ liệu: Gắn đặc trưng User vào lịch sử tương tác
print("🔄 Đang Join Dữ liệu Tương tác với User Features...")
df_train_full = df_interactions.join(df_user_features, on="user_id", how="inner")

# Bỏ bớt cột event_timestamp không cần thiết cho Training
if "event_timestamp" in df_train_full.columns:
    df_train_full = df_train_full.drop("event_timestamp")

# 5. Ghi dữ liệu trực tiếp lên thư mục Training Data trên GCS
output_path = "gs://amazon-reviews-lakehouse-warehouse/warehouse/gold/training_data"
print(f"💾 Đang ghi hàng chục triệu dòng Training Data xuống GCS tại: {output_path} ...")

df_train_full.repartition(32).write \
    .mode("overwrite") \
    .parquet(output_path)

end_time = time.time()
print(f"🎉 HOÀN THÀNH! Toàn bộ Training Data đã sẵn sàng trên Lakehouse.")
print(f"⏱️ Thời gian chạy: {round(end_time - start_time, 2)} giây.")

⏳ Khởi động Siêu máy tính Spark Single Node (16GB RAM)...


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/15 10:09:53 INFO SparkEnv: Registering MapOutputTracker
26/06/15 10:09:53 INFO SparkEnv: Registering BlockManagerMaster
26/06/15 10:09:53 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/06/15 10:09:53 INFO SparkEnv: Registering OutputCommitCoordinator


✅ Đã kết nối Lakehouse. Bắt đầu trộn dữ liệu...
⏳ Đang nạp User Features...


🔄 Đang Join Dữ liệu Tương tác với User Features...
💾 Đang ghi hàng chục triệu dòng Training Data xuống GCS tại: gs://amazon-reviews-lakehouse-warehouse/warehouse/gold/training_data ...
